# Dice vs no-Dice: test metrics analysis

This notebook compares test CSV metrics between two experiments (no-Dice vs Dice) and helps diagnose why CSI changes.
Adjust the experiment names and horizons to match your setup.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_rows", 200)

In [ ]:
base = Path("CSV_XP")
csvs = sorted(base.glob("Experience*.csv"))
csvs

In [ ]:
def load_csv(path):
    df = pd.read_csv(path)
    df["epoch"] = df["epoch"].astype(int)
    df["horizon_steps"] = df["horizon_steps"].astype(int)
    df["exp"] = path.stem
    return df

all_df = pd.concat([load_csv(p) for p in csvs], ignore_index=True)
all_df.head()

## Select experiments to compare

In [ ]:
exp_nodice = "Experience4"
exp_dice = "Experience5"
horizons = [6, 12, 24]  # 3h, 6h, 12h if dt=30min

df_nodice = all_df[all_df["exp"].eq(exp_nodice)].copy()
df_dice = all_df[all_df["exp"].eq(exp_dice)].copy()
df_nodice.head(), df_dice.head()

## Best CSI checkpoint per horizon

In [ ]:
def best_by_csi(df, horizons):
    rows = []
    for h in horizons:
        sub = df[df["horizon_steps"].eq(h)]
        if sub.empty:
            continue
        best = sub.loc[sub["csi"].idxmax()]
        rows.append({
            "horizon_steps": h,
            "epoch": int(best["epoch"]),
            "csi": float(best["csi"]),
            "mse_h": float(best["mse_h"]),
            "mse_u": float(best["mse_u"]),
            "mse_v": float(best["mse_v"]),
        })
    return pd.DataFrame(rows)

best_nodice = best_by_csi(df_nodice, horizons)
best_dice = best_by_csi(df_dice, horizons)
best_nodice, best_dice

## Align epochs and compute deltas (Dice - NoDice)

In [ ]:
merged = df_nodice.merge(
    df_dice,
    on=["epoch", "horizon_steps"],
    suffixes=("_nodice", "_dice"),
)
merged["dcsi"] = merged["csi_dice"] - merged["csi_nodice"]
merged["dmse_h"] = merged["mse_h_dice"] - merged["mse_h_nodice"]
merged["dmse_u"] = merged["mse_u_dice"] - merged["mse_u_nodice"]
merged["dmse_v"] = merged["mse_v_dice"] - merged["mse_v_nodice"]

merged[merged["horizon_steps"].isin(horizons)].groupby("horizon_steps")[
    ["dcsi", "dmse_h", "dmse_u", "dmse_v"]
].agg(["mean", "min", "max"])

## Plot CSI and MSE across epochs

In [ ]:
def plot_metric(metric, horizons):
    fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 3), sharey=False)
    if len(horizons) == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        a = df_nodice[df_nodice["horizon_steps"].eq(h)].sort_values("epoch")
        b = df_dice[df_dice["horizon_steps"].eq(h)].sort_values("epoch")
        ax.plot(a["epoch"], a[metric], marker="o", linewidth=2, label=exp_nodice)
        ax.plot(b["epoch"], b[metric], marker="o", linewidth=2, label=exp_dice)
        ax.set_title(f"{metric} @ h={h}")
        ax.set_xlabel("epoch")
        ax.grid(True, alpha=0.3)
    axes[0].legend()
    plt.tight_layout()
    plt.show()

plot_metric("csi", horizons)
plot_metric("mse_h", horizons)

## Delta scatter: do CSI gains come with MSE changes?

In [ ]:
fig, axes = plt.subplots(1, len(horizons), figsize=(5 * len(horizons), 3), sharey=True)
if len(horizons) == 1:
    axes = [axes]
for ax, h in zip(axes, horizons):
    sub = merged[merged["horizon_steps"].eq(h)]
    ax.scatter(sub["dmse_h"], sub["dcsi"], alpha=0.8)
    ax.axhline(0, color="black", linewidth=1)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(f"Delta CSI vs Delta MSE_h @ h={h}")
    ax.set_xlabel("dmse_h (dice - nodice)")
axes[0].set_ylabel("dcsi (dice - nodice)")
plt.tight_layout()
plt.show()

## Correlation between CSI and MSE inside each experiment

In [ ]:
def corr_table(df, horizons):
    rows = []
    for h in horizons:
        sub = df[df["horizon_steps"].eq(h)]
        if len(sub) < 2:
            continue
        rows.append({
            "horizon_steps": h,
            "corr_csi_mse_h": sub["csi"].corr(sub["mse_h"]),
            "corr_csi_mse_u": sub["csi"].corr(sub["mse_u"]),
            "corr_csi_mse_v": sub["csi"].corr(sub["mse_v"]),
        })
    return pd.DataFrame(rows)

corr_nodice = corr_table(df_nodice, horizons)
corr_dice = corr_table(df_dice, horizons)
corr_nodice, corr_dice

## Best/worst epochs for CSI delta

In [ ]:
for h in horizons:
    sub = merged[merged["horizon_steps"].eq(h)]
    if sub.empty:
        continue
    best = sub.loc[sub["dcsi"].idxmax()]
    worst = sub.loc[sub["dcsi"].idxmin()]
    print("h=", h)
    print(" best epoch", int(best["epoch"]), "dcsi", best["dcsi"], "dmse_h", best["dmse_h"])
    print(" worst epoch", int(worst["epoch"]), "dcsi", worst["dcsi"], "dmse_h", worst["dmse_h"])
    print()

## Wetness diagnostics (bias / wet fraction / false positives)

This section recomputes diagnostics directly from raw predictions to test the "over-wetting" hypothesis.
Update the paths below to your test PKLs and checkpoint folders before running.


In [ ]:
import os, sys, math
import torch, dgl
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.append(ROOT)

from python.create_dgl_dataset import TelemacDataset
from python.CustomMeshGraphNet import MeshGraphNet
from modulus.launch.utils import load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
# =====================
# Parametres utilisateur (a adapter)
# =====================

DATA_DIR = "/work/m24046/m24046mrcr/paper/Experience2/Multimesh_8_32.bin"
DYNAMIC_DIR = [
    # ajoute ici les PKL du test
]

HORIZONS_STEPS = list(horizons) if 'horizons' in globals() else [12, 24]
MAX_SEQUENCES = 205
THRESHOLD_M = 0.05
OVERLAP = 1
MASK_BOUNDARY = True  # True = on exclut les BC des stats wet/bias

# Parametres modele (coherents avec l'entrainement)
NUM_INPUT_FEATURES = 9
NUM_EDGE_FEATURES = 3
NUM_OUTPUT_FEATURES = 3
MP_LAYERS = 10
DO_CONCAT_TRICK = True
NUM_PROCESSOR_CHECKPOINT_SEGMENTS = 0

# Checkpoints a comparer
exp_nodice_name = exp_nodice if 'exp_nodice' in globals() else 'nodice'
exp_dice_name = exp_dice if 'exp_dice' in globals() else 'dice'

EXPERIMENTS = {
    exp_nodice_name: {
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience3/Seed0/',
        'epochs': [600, 650, 700, 750, 800],
    },
    exp_dice_name: {
        'ckpt_dir': '/work/m24046/m24046mrcr/paper/Experience5_1/Seed0/',
        'epochs': [600, 650, 700, 750, 800],
    },
}


In [ ]:
# =====================
# Fonctions utilitaires
# =====================
def build_model():
    return MeshGraphNet(
        NUM_INPUT_FEATURES,
        NUM_EDGE_FEATURES,
        NUM_OUTPUT_FEATURES,
        processor_size=MP_LAYERS,
        hidden_dim_processor=64,
        hidden_dim_node_encoder=64,
        hidden_dim_edge_encoder=64,
        hidden_dim_node_decoder=64,
        do_concat_trick=DO_CONCAT_TRICK,
        num_processor_checkpoint_segments=NUM_PROCESSOR_CHECKPOINT_SEGMENTS,
    )

def load_model_checkpoint(model, ckpt_dir, epoch):
    load_checkpoint(ckpt_dir, models=model, device=device, epoch=epoch)
    model.to(device)
    model.eval()
    return model

def build_dataset(sequence_length, overlap, ckpt_dir, split="test"):
    return TelemacDataset(
        name=f"eval_{split}",
        data_dir=DATA_DIR,
        dynamic_data_files=DYNAMIC_DIR,
        split=split,
        ckpt_path=ckpt_dir,
        normalize=True,
        sequence_length=sequence_length,
        overlap=overlap,
    )

def _denorm(xn, mean, std):
    return xn * std + mean

def _renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)

def evaluate_model_wetness(model, ds, horizons_steps, max_sequences=10, threshold=0.05, mask_boundary=True):
    stats = ds.node_stats
    dyn_start = ds.base_graph.ndata['static'].shape[1]
    mx = torch.tensor([stats['h'].item(), stats['u'].item(), stats['v'].item()], device=device)
    sx = torch.tensor([stats['h_std'].item(), stats['u_std'].item(), stats['v_std'].item()], device=device)
    dy_mean = torch.tensor([stats['delta_h'].item(), stats['delta_u'].item(), stats['delta_v'].item()], device=device)
    dy_std  = torch.tensor([stats['delta_h_std'].item(), stats['delta_u_std'].item(), stats['delta_v_std'].item()], device=device)

    agg = {h: {
        'count': 0,
        'bias_sum': 0.0,
        'wet_pred_sum': 0,
        'wet_gt_sum': 0,
        'fp_sum': 0,
        'fn_sum': 0,
    } for h in horizons_steps}

    nseq = min(max_sequences, len(ds))
    max_h = max(horizons_steps)

    for idx in range(nseq):
        graphs = ds[idx]
        if len(graphs) <= max_h:
            continue
        g = graphs[0].to(device)
        static_part = g.ndata['x'][:, :dyn_start]
        xn_t = g.ndata['x'][:, dyn_start:dyn_start+3]

        onehot = static_part[:, :4]
        q_mask = (onehot == torch.tensor([0,0,1,0], device=device)).all(dim=1)
        h_mask = (onehot == torch.tensor([0,1,0,0], device=device)).all(dim=1)
        interior_mask = ~(q_mask | h_mask) if mask_boundary else torch.ones_like(q_mask)

        for t in range(max_h):
            with torch.no_grad():
                y_pred_n = model(g.ndata['x'], g.edata['x'], g)
            x_t = _denorm(xn_t, mx, sx)
            y_pred = _denorm(y_pred_n, dy_mean, dy_std)
            x_t1 = x_t + y_pred

            x_gt_n = graphs[t+1].ndata['x'][:, dyn_start:dyn_start+3].to(device)
            x_gt = _denorm(x_gt_n, mx, sx)

            # Conditions limites
            x_t1[q_mask] = x_gt[q_mask]
            x_t1[h_mask, 0:1] = x_gt[h_mask, 0:1]

            step = t + 1
            if step in horizons_steps:
                h_pred = x_t1[:, 0][interior_mask]
                h_gt = x_gt[:, 0][interior_mask]
                if h_pred.numel() == 0:
                    continue

                h_pred_np = h_pred.detach().cpu().numpy()
                h_gt_np = h_gt.detach().cpu().numpy()

                pred_wet = h_pred_np >= threshold
                gt_wet = h_gt_np >= threshold

                n = h_pred_np.size
                agg[step]['count'] += n
                agg[step]['bias_sum'] += float((h_pred_np - h_gt_np).sum())
                agg[step]['wet_pred_sum'] += int(pred_wet.sum())
                agg[step]['wet_gt_sum'] += int(gt_wet.sum())
                agg[step]['fp_sum'] += int(np.logical_and(pred_wet, ~gt_wet).sum())
                agg[step]['fn_sum'] += int(np.logical_and(~pred_wet, gt_wet).sum())

            # reinjection
            xn_t = _renorm(x_t1, mx, sx)
            g = g.clone()
            g.ndata['x'] = torch.cat([static_part, xn_t], dim=1)

    summary = {}
    for h in horizons_steps:
        count = agg[h]['count']
        if count == 0:
            continue
        wet_pred = agg[h]['wet_pred_sum'] / count
        wet_gt = agg[h]['wet_gt_sum'] / count
        summary[h] = {
            'bias_mean': agg[h]['bias_sum'] / count,
            'wet_frac_pred': wet_pred,
            'wet_frac_gt': wet_gt,
            'wet_frac_delta': wet_pred - wet_gt,
            'false_pos_rate': agg[h]['fp_sum'] / count,
            'false_neg_rate': agg[h]['fn_sum'] / count,
        }
    return summary

def run_wet_eval(experiments):
    rows = []
    seq_len = max(HORIZONS_STEPS) + 1
    for exp_name, cfg in experiments.items():
        ckpt_dir = cfg['ckpt_dir']
        epochs = cfg['epochs']
        ds = build_dataset(sequence_length=seq_len, overlap=OVERLAP, ckpt_dir=ckpt_dir)
        for ep in epochs:
            model = build_model()
            load_model_checkpoint(model, ckpt_dir, epoch=ep)
            summary = evaluate_model_wetness(
                model,
                ds,
                HORIZONS_STEPS,
                max_sequences=MAX_SEQUENCES,
                threshold=THRESHOLD_M,
                mask_boundary=MASK_BOUNDARY,
            )
            for h, vals in summary.items():
                rows.append({
                    'exp': exp_name,
                    'epoch': ep,
                    'horizon_steps': h,
                    **vals,
                })
    return pd.DataFrame(rows)


In [ ]:
wet_df = run_wet_eval(EXPERIMENTS)
wet_df.sort_values(["exp", "horizon_steps", "epoch"]).head()


In [ ]:
def plot_wet_metrics(df, horizons_steps):
    for h in horizons_steps:
        sub = df[df["horizon_steps"].eq(h)]
        if sub.empty:
            continue
        fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharex=True)
        for exp_name, g in sub.groupby("exp"):
            g = g.sort_values("epoch")
            axes[0].plot(g["epoch"], g["wet_frac_pred"], marker="o", label=f"{exp_name} pred")
            axes[0].plot(g["epoch"], g["wet_frac_gt"], marker="x", linestyle="--", label=f"{exp_name} gt")
            axes[1].plot(g["epoch"], g["bias_mean"], marker="o", label=exp_name)
            axes[2].plot(g["epoch"], g["false_pos_rate"], marker="o", label=exp_name)

        axes[0].set_title(f"Wet fraction (h={h})")
        axes[1].set_title(f"Bias mean (h={h})")
        axes[2].set_title(f"False positive rate (h={h})")

        for ax in axes:
            ax.grid(True, alpha=0.3)
            ax.set_xlabel("epoch")
        axes[0].legend(loc="best", fontsize=8)
        axes[1].legend(loc="best", fontsize=8)
        axes[2].legend(loc="best", fontsize=8)
        plt.tight_layout()
        plt.show()

plot_wet_metrics(wet_df, HORIZONS_STEPS)
